In [1]:
import numpy as np
import torch
%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

# Carregar o modelo e as referências uma única vez.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo usado: {device}")
model = SentenceTransformer(
    'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    device=device,
)
def_processo = open("Processo_Geral.csv", "r", encoding="utf-8").read().strip()
def_produto = open("Produto_Geral.csv", "r", encoding="utf-8").read().strip()
emb_referencias = model.encode(
    [def_processo, def_produto],
    normalize_embeddings=True,
)

Note: you may need to restart the kernel to use updated packages.


c:\Users\ECO\Desktop\ITI - Polarização\Patentes\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
df = pd.read_csv("df_automacao_limpo.csv", sep=';')
df['ipc_codes'] = df['ipc_codes'].str.split(';').str[0].str.strip()

In [3]:
df

,publication_number,filing_date,ipc_codes,title,abstract
0,BR-112020007064-A2,20180620,D04H3/16; D06M17/00; A61F13/49; D04H3/007; B32...,folha elástica para artigo absorvente e artigo...,A presente invenção refere-se a folha elástica...
1,BR-112013006594-B1,20110920,C07D239/88; A61P25/24; C07D409/14; C07D471/04;...,Composto ciclopropano antagonista de receptor ...,"COMPOSTO, COMPOSIÇÃO FARMACÊUTICA, MÉTODO DE T..."
2,BR-112021026164-A2,20200625,G07D11/13; G07F19/00; G07D11/18,Máquina de transação automatizada e cassete pa...,MÁQUINA DE TRANSAÇÃO AUTOMATIZADA E CASSETE PA...
3,BR-112018070089-A2,20170331,H04L1/00,métodos de operação de um dispositivo sem fio ...,são providos sistemas e métodos para relato da...
4,BR-202021008981-U2,20210507,A22C7/00,Equipamento automático para montar espetinhos ...,EQUIPAMENTO AUTOMÁTICO PARA MONTAR ESPETINHOS ...
...,...,...,...,...,...
384322,BR-112015008130-A8,20131008,G06F3/041,Sistema de tela sensível ao toque em moldura s...,SISTEMA DE TELA SENSÍVEL AO TOQUE EM MOLDURA S...
384323,BR-102017018829-A8,20170901,G06Q30/06,Sistema de busca de produtos e serviços com in...,SISTEMA DE BUSCA DE PRODUTOS E SERVIÇOS COM IN...
384324,BR-112015022632-B1,20140311,C22C21/00,"Composição de liga de magnésio, implante, e mé...",resumo patente de invenção &#34;liga de magnés...
384325,BR-112015028985-A2,20140521,B01D39/20,elemento de filtro cerâmico e método de fabric...,1/1 resumo elemento de filtro cerâmico e méto...


In [ ]:
from tqdm.auto import tqdm

textos = df["abstract"].fillna("").astype(str).tolist()
resultados = []
batch_size = 128 if device == "cuda" else 32

with tqdm(
    total=len(textos),
    desc="Processando patentes",
    unit="patente",
) as progresso:
    for inicio in range(0, len(textos), batch_size):
        lote = textos[inicio:inicio + batch_size]
        emb_patentes = model.encode(
            lote,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        similaridades = emb_patentes @ emb_referencias.T

        for texto, (sim_processo, sim_produto) in zip(lote, similaridades):
            resultados.append({
                "Classificação": (
                    "Inovação de Processo"
                    if sim_processo > sim_produto
                    else "Inovação de Produto"
                ),
                "Similaridade com Processo": float(sim_processo),
                "Similaridade com Produto": float(sim_produto),
                "Abstrato": texto,
            })
        progresso.update(len(lote))

resultado = pd.DataFrame(resultados)
resultado.head()

Processando patentes:   0%|          | 0/384327 [00:00<?, ?patente/s]


Analisando patente: A presente invenção refere-se a folha elástica par...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1407.48it/s]


Classificação: Inovação de Processo, Similaridade com Processo: 0.4757, Similaridade com Produto: 0.3878


,Classificação,Similaridade com Processo,Similaridade com Produto,Abstrato
0,Inovação de Processo,0.475666,0.387806,A presente invenção refere-se a folha elástica...


Processando patentes:   0%|          | 1/384327 [00:03<321:44:04,  3.01s/patente]

... (21 outputs left out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 928.20it/s]


Classificação: Inovação de Processo, Similaridade com Processo: 0.4106, Similaridade com Produto: 0.3390


,Classificação,Similaridade com Processo,Similaridade com Produto,Abstrato
0,Inovação de Processo,0.410612,0.33897,plataformas de exibição de endósporos intergen...


Processando patentes:   0%|          | 6/384327 [00:19<355:48:42,  3.33s/patente]